<div style="background-image: url('https://www.dropbox.com/scl/fi/nwtuyf23tzyuq9gfan3oc/mcnair.jpg?rlkey=4t1of5cy6bfgxn6xck7yu9hvo&dl=1'); background-size: cover; background-position: center; height: 300px; display: flex; align-items: center; justify-content: center; color: white; text-shadow: 2px 2px 4px rgba(0,0,0,0.7); margin-bottom: 20px; position: relative;">
  <h1 style="text-align: center; font-size: 2.5em; margin: 0;">JGSB Python Workshop <br> Part 10: More Visualization</h1>
  <div style="position: absolute; bottom: 10px; left: 15px; font-size: 0.9em; color: white; text-shadow: 2px 2px 4px rgba(0,0,0,0.7);">
    Authored by Kerry Back
  </div>
  <div style="position: absolute; bottom: 10px; right: 15px; text-align: right; font-size: 0.9em; color: white; text-shadow: 2px 2px 4px rgba(0,0,0,0.7);">
    Rice University, 9/6/2025
  </div>
</div>

### Wages Data

We'll use the wages data again.

In [ ]:
# Read data and create DataFrame
 
url = 'https://faculty.utrgv.edu/diego.escobari/teaching/Datasets/WAGE1.xls'
wages = pd.read_excel(url, header=None)

columns = [
    'wage', 'educ', 'exper',
    'tenure', 'nonwhite', 'female',
    'married', 'numdep',
    'smsa', 'northcen',
    'south', 'west', 'construc',
    'ndurman', 'trcommpu', 'trade',
    'services', 'profserv',
    'profocc', 'clerocc', 'servocc',
    'lwage', 'expersq', 'tenursq'
]
wages.columns = columns

# Convert binary variables to categorical variables for better visualization
# This makes the data more interpretable for plotting

wages['Gender'] = wages['female'].map({0: 'Male', 1: 'Female'})
wages['Race'] = wages['nonwhite'].map({0: 'White', 1: 'Non-white'})
wages['Marital_Status'] = wages['married'].map({0: 'Not Married', 1: 'Married'})
wages['Urban'] = wages['smsa'].map({0: 'Rural', 1: 'Urban'})

# Create region categories
def get_region(row):
    if row['northcen'] == 1:
        return 'North Central'
    elif row['south'] == 1:
        return 'South'
    elif row['west'] == 1:
        return 'West'
    else:
        return 'Northeast'

wages['Region'] = wages.apply(get_region, axis=1)

# Create industry categories
def get_industry(row):
    if row['construc'] == 1:
        return 'Construction'
    elif row['ndurman'] == 1:
        return 'Non-durable Manufacturing'
    elif row['trcommpu'] == 1:
        return 'Transportation/Communications'
    elif row['trade'] == 1:
        return 'Trade'
    elif row['services'] == 1:
        return 'Services'
    elif row['profserv'] == 1:
        return 'Professional Services'
    else:
        return 'Other'

wages['Industry'] = wages.apply(get_industry, axis=1)

# Create occupation categories
def get_occupation(row):
    if row['profocc'] == 1:
        return 'Professional'
    elif row['clerocc'] == 1:
        return 'Clerical'
    elif row['servocc'] == 1:
        return 'Service'
    else:
        return 'Other'

wages['Occupation'] = wages.apply(get_occupation, axis=1)

# Create education categories
wages['Education_Level'] = pd.cut(wages['educ'], 
                                 bins=[0, 12, 16, 25], 
                                 labels=['High School or Less', 'Some College', 'College Plus'])

# Create experience categories  
wages['Experience_Level'] = pd.cut(wages['exper'], 
                                  bins=[0, 5, 15, 60], 
                                  labels=['Low (0-5)', 'Mid (6-15)', 'High (16+)'])

# Delete columns we do not need
wages = wages.drop(columns=wages.columns[4:-9])

# View top of data
wages.head()

### Seaborn `pairplot`

A seaborn pairplot is a matrix of scatter plots.  

- Ask Gemini to create a seaborn pairplot with `corner=True` of all numeric variables in the wages DataFrame. 

- The plots on the diagonal can be histograms or density plots.  Ask Gemini to produce both.

- Ask Gemini to include regression lines in the scatter plots.

- Ask Gemini to try different transparency settings.

### Visualizing the Effects of Explanatory Variables

  We want to visually explore what
  determines wages. We want to understand:
  - Which factors most strongly influence
  wages?
  - How do different variables interact to
   affect wages?
  - Are there patterns or outliers we
  should investigate?

  Wages is our **target variable**
  (numeric). Other variables are
  **explanatory variables** (numeric or
  categorical).

  #### Single Explanatory Variable 

  **Numeric explanatory variable:**
  - **Scatter plot**: Shows relationship
  and correlation
  - **Regression plot**: Adds trend line
  to see direction and strength

  **Categorical explanatory variable:**
  - **Box plot**: Shows distribution of
  wages within each category (preferred)
  - **Strip plot**: Alternative to box plot to show each individual data point
  - **Bar plot**: Show means by
  category (good for summary).  Add `errorbar` argument to show dispersion of data by category
  - **Violin plot**: Shows both
  distribution shape and summary
  statistics
  

  #### Multiple Explanatory Variables

  **Multiple categorical variables:**
  - **Heatmap**: Matrix of two category combinations with color representing mean wage.
  - **Grouped bar or box plots:** Use y = wage, x = first categorical variable, and use `hue` for second categorical variable.
  - **Faceted box plots:** Matrix of box plots, with one categorical variable across columns and another across rows.
 

  **Numeric + Categorical:**
  - **Scatter plot**: Use `hue`, `size`, or `style` to distinguish different values of the categorical variable.  `hue` is usually the best choice for a single categorical variable.  Use `size` or `style` for a second categorical variable.  More than two categorical variables is usually unreadable.
  - **Faceted scatter plots**: With one numeric and two categorical variables, the clearest plot is usually a matrix of scatter plots, with one categorical variable across columns and another across rows.
  

  **Two numeric variables:**
  - **Scatter plot**: Use x = first categorical variable, y = second categorical variable, and `hue` or `size` for wage.  
  - **3D scatter plot**: x, y, z
  coordinates (use sparingly, difficult to understand)

  **Choose the right plot:**
  - Start with **box plots** for
  categorical variables (most informative)
  - Use **scatter plots** for numeric
  relationships.  It is usually a good idea to include a regression line.
  - Add **hue/size/style** to show effects of
  additional variables

  **Make it interpretable:**
  - Always label axes clearly ("Annual
  Wage ($)", not just "wage")
  - Use meaningful titles ("Wage by
  Education Level and Gender")
  - Order categories logically (Low →
  High, not alphabetical)
  - Consider log scale if wage values span
   wide ranges

  **Look for patterns:**
  - Non-linear relationships (curved
  scatter plots)
  - Outliers (unusual wage values).  May be useful to annotate outliers.
  - Interactions (effect of one variable
  depends on another)

**Recommended Seaborn Functions**

  - `sns.boxplot()` - Best for categorical
  - `sns.scatterplot()` - Best for numeric
  - `sns.regplot()` - Adds trend line to
  scatter plots
  - Use `hue`, `size`, `style` parameters
  to add additional variables
  - `sns.heatmap()` - Good for two
  categorical variables
  - `sns.catplot()` - Matrix of bar plots, box plots, violin plots, strip plots, or swarm plots
   - `sns.replot()` - Matrix of scatter plots
  


### Exercises 

- For various numeric and categorical explanatory variables, ask Gemini to create seaborn plots showing their effects on wage.  
- Ask Gemini for recommendations and also ask Gemini to produce several alternatives so you can decide which is best.
- If Gemini does not use one of the plot types listed in the previous cell, ask for it specifically so you can compare it to others.
- When using `hue`, ask Gemini to try different palettes.
- Scatter plots can be difficult to comprehend when there are many data points.  Two solutions are:
  - (1) Make points partially transparent.  Darker regions are then regions with more points.
  - (2) Bin the x variable into buckets.  Plot the means of x and y within each bucket.
  - Ask Gemini to use `sns.regplot` with different values of `alpha` (for transparency) and `x_bins` (for the number of x bins).